# Investigating the Met's Collection
### What Gets Displayed, What Gets Collected, and What Art Reveals About Its Era

DS3000 Final Project

Group: Anna Bennett, Zara Ceraj

**1. (20 pts) Expresses the central motivation of the project and explains the (at least) two key questions to be explored. Gives a summary of the data processing pipeline so a technical expert can easily follow along.**

### Central Motivation and Key Questions
When you walk into a large art museum, it feels like you could spend a week walking through carefully looking at every piece. However, most museums only show a fraction of their collections. For instance, "the Tate shows about 20% of its permanent collection. The Louvre shows 8%, the Guggenheim a lowly 3% and the Berlinische Galerie – a Berlin museum whose mandate is to show, preserve and collect art made in the city – 2% of its holdings" (Bradley, 2015). Physical space is part of the reason, but what ends up on the walls also reflects each institution's curatorial priorities. Our questions arise out of these inaccessible collections that many museums have, focusing more specifically on the Metropolitan Museum of Art.

Question 1: What predicts whether a work is on display?

Can features like an artwork's age, size, department, culture of origin, or the artist's gender predict whether it hangs in a gallery or sits in storage?

What gets displayed depends on what was collected in the first place, and collecting has historically been uneven. In 1989, the Guerrilla Girls, an anonymous collective of feminist artists, visited the Met and compared the number of women artists in its modern art galleries with the number of female nudes on display. They found that fewer than 5% of the artists in those galleries were women, while 85% of the nudes were female (Guerrilla Girls, 1989). Decades later, the gap remains: between 2008 and 2018, "works by women constituted just 11 percent of acquisitions and 14 percent of exhibitions at 26 major American museums" (Solly, 2019).

Artistic movements are often defined by new materials, techniques, or philosophies, but there is overlap across very different eras. Oil paint, for example, was used centuries apart by both Renaissance and Impressionist painters.

Question 2: How accurately can an artwork's time period be predicted from its traits, such as its medium, size, and culture of origin?

### Data Processing Pipeline
1. **Get IDs:** Request all object IDs (~502,000) from the Met API's `/objects` endpoint.
2. **Sample:** Randomly select 500 IDs (`random.seed(3000)`), since the API returns one object per request.
3. **Fetch:** Request each object from `/objects/{id}`, retrying failed requests up to 3 times with increasing waits and pausing 1 s between requests. Objects that still fail are skipped.
4. **Extract:** Keep 14 fields per object, including department, culture, medium, artist, dates, accession year, and gallery number. Height and width (cm) come from the `Overall` measurement and are left blank if missing.
5. **Save:** Combine the rows into a DataFrame (500 × 15) and save to `met_sample.csv`, which all later steps load instead of calling the API.
6. **Clean (`clean_df`, 500 × 23):**
   - Set impossible values to missing (a 2056 accession year and a 0/0 placeholder date).
   - Create the Question 1 target: `onView` is True when a work has a gallery number.
   - Fill missing cultures with "Unknown" and merge duplicate labels (e.g., "Japan" → "Japanese").
   - Group the 286 free-text mediums into broad categories (`medium_group`) and combine departments with fewer than 10 works into "Other".
   - Add flags for whether a work has measurements (`has_size`), a named artist (`has_artist`), or a female artist (`artist_female`; the gender field is otherwise blank, meaning "not recorded").
   - Bin creation year into periods (`period`) 


## Data Source, obtaining it, and cleaning it
**2. (25 pts) Obtains, cleans, and merges all data sources involved in the project.**

### Data Source
The Metropolitan Museum of Art (the Met) in New York City is one of the largest art museums in the world, with a collection spanning thousands of years. The museum publishes data on over 502,000 artworks through a free public API that requires no registration or key.

Each artwork's record includes dozens of fields, such as when the work was made, when the museum acquired it, its dimensions, its department and culture of origin, and whether it is currently on view.

[Data Source: The Metropolitan Museum of Art Collection API](https://metmuseum.github.io/)

In [123]:
import random
import time

import pandas as pd
import requests

In [124]:
# Already run once and saved to met_sample.csv -- uncomment to re-fetch from the API
# url = "https://collectionapi.metmuseum.org/public/collection/v1/objects"

# response = requests.get(url)

# all_ids = response.json()["objectIDs"]
# print("Total objects in the collection:", len(all_ids))

# random.seed(3000)
# sample_ids = random.sample(all_ids, 500)

Note: Due to the size of the collection and since the API only returns one object per request, we took a random sample of 500 IDs. This sample might not accurately reflect the full collection, but the API's rate limits made larger-scale collection impractical.

In [125]:
# Already run once and saved to met_sample.csv -- uncomment to re-fetch from the API (slow).
# rows = []
# failed = 0

# for object_id in sample_ids:
#     # retry up to 3 times, waiting longer each time, since the API blocks fast repeated requests
#     response = None
#     for attempt in range(3):
#         response = requests.get(url + "/" + str(object_id), headers={"User-Agent": "Mozilla/5.0"})
#         if response.status_code == 200:
#             break
#         time.sleep(5 * (attempt + 1))
#     if response.status_code != 200:
#         failed += 1
#         continue
#     obj = response.json()

#     # find the overall height and width but some have missing data
#     height = None
#     width = None
#     if obj["measurements"] is not None:
#         for m in obj["measurements"]:
#             if m["elementName"] == "Overall":
#                 sizes = m["elementMeasurements"]
#                 if "Height" in sizes:
#                     height = sizes["Height"]
#                 if "Width" in sizes:
#                     width = sizes["Width"]

#     row = {
#         "objectID": obj["objectID"],
#         "title": obj["title"],
#         "department": obj["department"],
#         "classification": obj["classification"],
#         "culture": obj["culture"],
#         "medium": obj["medium"],
#         "artistDisplayName": obj["artistDisplayName"],
#         "artistGender": obj["artistGender"],
#         "objectBeginDate": obj["objectBeginDate"],
#         "objectEndDate": obj["objectEndDate"],
#         "accessionYear": obj["accessionYear"],
#         "height_cm": height,
#         "width_cm": width,
#         "GalleryNumber": obj["GalleryNumber"],
#     }
#     rows.append(row)

#     time.sleep(1) # had to keep extending this...

# print("Failed requests:", failed)

# met_df = pd.DataFrame(rows)
# met_df.head()

In [126]:
# load the saved sample instead of calling the API every time
met_df = pd.read_csv("met_sample.csv")
met_df.head()

,objectID,title,department,classification,culture,medium,artistDisplayName,artistGender,objectBeginDate,objectEndDate,accessionYear,height_cm,width_cm,GalleryNumber,onView
0,502128,Sek Kei,Musical Instruments,Idiophone-Struck-gong,Japanese,"Stone, wood",NaN,NaN,1801,1900,1889.0,NaN,NaN,NaN,False
1,81088,Dress,Costume Institute,NaN,Belgian,leather,Ann Demeulemeester,NaN,1997,1997,1998.0,NaN,NaN,NaN,False
2,268387,St. Andrews. The Pends,Photographs,Photographs,NaN,Salted paper print from paper negative,Hill and Adamson,NaN,1843,1847,1937.0,NaN,NaN,NaN,False
3,51453,Bird Pendant,Asian Art,Jewelry,"Indonesia (Java, Lumajang, Pasiran)",Bronze,NaN,NaN,-500,300,2001.0,2.10,3.5,NaN,False
4,248894,Bronze box mirror,Greek and Roman Art,Bronzes,Greek,Bronze,NaN,NaN,-330,-300,1914.0,14.81,NaN,158.0,True


In [127]:
# cleaning and simplifying the data
# onView is already in the CSV (blank gallery numbers load as NaN, so != "" would mark everything True)
# met_df["onView"] = met_df["GalleryNumber"] != ""
met_df["accessionYear"] = pd.to_numeric(met_df["accessionYear"], errors="coerce")

# met_df.to_csv("met_sample.csv", index=False)

print("DataFrame shape:", met_df.shape)

print("\nDescription:")
met_df.describe()
met_df.sample(10)

DataFrame shape: (500, 15)

Description:


,objectID,title,department,classification,culture,medium,artistDisplayName,artistGender,objectBeginDate,objectEndDate,accessionYear,height_cm,width_cm,GalleryNumber,onView
345,710360,Textile Design with Scrolling Branches and Pearls,Drawings and Prints,Drawings,NaN,Gouache,"Anonymous, Alsatian, 19th century",NaN,1840,1840,1978.0,NaN,NaN,NaN,False
427,893911,Four Wheel Dog Cart #4023,Drawings and Prints,Drawings,NaN,graphite,Brewster & Co.,NaN,1888,1888,1923.0,NaN,NaN,NaN,False
67,764026,Portrait of La Live de Jully,Drawings and Prints,Prints,NaN,Etching,Ange-Laurent de La Live de Jully,NaN,1740,1780,2014.0,NaN,NaN,NaN,False
487,346841,The Pilgrim's Progress: From This World to Tha...,Drawings and Prints,Books,NaN,Illustrations: wood engraving,John Bunyan,NaN,1880,1880,1969.0,28.2576,22.5425,NaN,False
129,668692,Massacre of the Innocents; group of women and ...,Drawings and Prints,Prints,NaN,Etching,Gian Battista Bolognini,NaN,1635,1670,1951.0,NaN,NaN,NaN,False
341,400230,"Ewing and Mascot, New York, from the series Ol...",Drawings and Prints,Photographs,NaN,"Albumen photograph, cabinet card",Goodwin & Company,NaN,1887,1887,1963.0,NaN,NaN,NaN,False
187,924844,Partial design for a pendant or a brooch (?),Drawings and Prints,Drawings,NaN,Graphite,Fernand Chardon,NaN,1920,1930,1980.0,NaN,NaN,NaN,False
142,709957,Terracotta fragment of a kylix (drinking cup),Greek and Roman Art,Vases,"Greek, Attic",Terracotta,Antiphon Painter,NaN,-530,-450,2011.0,NaN,NaN,NaN,False
41,323778,Axe head,Ancient West Asian Art,NaN,Sumerian,Bronze,NaN,NaN,-2900,-2350,1941.0,NaN,NaN,NaN,False
175,21875,Blade and Mounting for a Dagger (Tantō),Arms and Armor,Daggers,Japanese,"Steel, wood, lacquer, copper-gold alloy (shaku...",NaN,NaN,1501,1900,1891.0,NaN,NaN,NaN,False


Note: the dates make sense because they used negative dates for BC and positive for AD.

#### Code Cleaning 

In [128]:
import numpy as np

clean_df = met_df.copy()

# 1. Fix impossible / placeholder values
clean_df.loc[clean_df["accessionYear"] > 2026, "accessionYear"] = np.nan
unknown_date = (clean_df["objectBeginDate"] == 0) & (clean_df["objectEndDate"] == 0)
clean_df.loc[unknown_date, ["objectBeginDate", "objectEndDate"]] = np.nan

# 2. Rebuild onView here so the pipeline is explicit (blank GalleryNumber -> not on view)
clean_df["onView"] = clean_df["GalleryNumber"].notna()

# 3. Culture: fill blanks, merge country/adjective duplicates
clean_df["culture"] = clean_df["culture"].fillna("Unknown")
clean_df["culture"] = clean_df["culture"].replace({"Japan": "Japanese", "China": "Chinese"})
clean_df["culture_group"] = clean_df["culture"].str.split(",").str[0].str.strip()

# 4. Medium: map 286 free-text values to broad material groups
# AI usage: claude used to come up with these groups 
def medium_group(m):
    if pd.isna(m):
        return "Unknown"
    m = m.lower()
    groups = {
        "Photograph": ["photograph", "gelatin", "albumen", "negative", "salted paper", "platinum"],
        "Print": ["lithograph", "etching", "engraving", "woodcut", "print"],
        "Drawing/Paint": ["ink", "graphite", "chalk", "watercolor", "oil", "tempera", "gouache"],
        "Ceramic": ["terracotta", "ceramic", "porcelain", "earthenware", "stoneware", "faience"],
        "Metal": ["bronze", "silver", "gold", "copper", "iron", "steel", "brass"],
        "Textile": ["silk", "cotton", "wool", "linen", "leather"],
        "Stone/Glass": ["stone", "marble", "limestone", "glass"],
        "Wood": ["wood"],
    }
    for group, words in groups.items():
        if any(w in m for w in words):
            return group
    return "Other"

clean_df["medium_group"] = clean_df["medium"].apply(medium_group)

# 5. Size: keep rows, add a flag + area
clean_df["has_size"] = clean_df["height_cm"].notna() & clean_df["width_cm"].notna()
clean_df["area_cm2"] = clean_df["height_cm"] * clean_df["width_cm"]

# 6. Artist info as flags
clean_df["has_artist"] = clean_df["artistDisplayName"].notna()
clean_df["artist_female"] = clean_df["artistGender"] == "Female"

# 7. Collapse small departments
dept_counts = clean_df["department"].value_counts()
small = dept_counts[dept_counts < 10].index
clean_df["department_group"] = clean_df["department"].replace(small, "Other")

# 8. Bin creation date into periods
bins   = [-np.inf, -500, 500, 1400, 1600, 1800, 1900, np.inf]
labels = ["Ancient (<500 BC)", "Classical (500 BC–500 AD)", "Medieval", "Renaissance",
          "Early Modern", "19th Century", "20th Century+"]
clean_df["period"] = pd.cut(clean_df["objectBeginDate"], bins=bins, labels=labels)

print(clean_df.shape)
print(clean_df.isna().sum())
clean_df["medium_group"].value_counts()


(500, 23)
objectID               0
title                  0
department             0
classification        93
culture                0
medium                 9
artistDisplayName    195
artistGender         489
objectBeginDate        1
objectEndDate          1
accessionYear          8
height_cm            320
width_cm             326
GalleryNumber        457
onView                 0
culture_group          0
medium_group           0
has_size               0
area_cm2             344
has_artist             0
artist_female          0
department_group       0
period                 1
dtype: int64


medium_group
Print            135
Metal             61
Other             59
Textile           55
Drawing/Paint     50
Ceramic           49
Photograph        45
Stone/Glass       28
Wood               9
Unknown            9
Name: count, dtype: int64

## Visualizations
**3. (25 pts) Builds at least two visualizations (graphs/plots) from the data which help to understand or answer the questions of interest. These visualizations will be graded based on how much information they can effectively communicate to readers. Please make sure your visualization are sufficiently distinct from each other.**

In [129]:
# calculate display percentage by department and classification
import plotly.express as px

# find the 15 departments with the most artworks
top_departments = clean_df["department"].value_counts().head(15).index
plot_data = clean_df[clean_df["department"].isin(top_departments)].copy()

# count displayed and undisplayed artworks
plot_data["Display Status"] = plot_data["onView"].map({
    True: "On Display",
    False: "Not on Display"})

fig = px.histogram(plot_data,
    x="department",
    color="Display Status",
    barmode="group",
    title="Displayed vs. Undisplayed Artworks by Department",
    labels={"department": "Museum Department",
        "count": "Number of Artworks"})

fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [ ]:
# visualization that compares when each work was made vs. when the Met acquired it
scatter_data = clean_df.dropna(subset=["objectBeginDate", "accessionYear"]).copy()
scatter_data["Display Status"] = scatter_data["onView"].map({True: "On Display", False: "Not on Display"})

pct_bc = scatter_data[scatter_data["objectBeginDate"] < 0]["onView"].mean() * 100
pct_ad = scatter_data[scatter_data["objectBeginDate"] >= 0]["onView"].mean() * 100

fig = px.scatter(scatter_data,
    x="objectBeginDate",
    y="accessionYear",
    color="Display Status",
    hover_data=["title", "department", "medium"],
    opacity=0.7,
    title="When Works Were Made vs. When the Met Acquired Them",
    labels={"objectBeginDate": "Year Created",
        "accessionYear": "Year Acquired by the Met"})

ticks = [-4000, -3000, -2000, -1000, 0, 1000, 2000]

fig.update_xaxes(tickvals=ticks, ticktext=[f"{-t} BC" if t < 0 else f"{t} AD" for t in ticks],
    range=[-4100, 2100], showgrid=False, zeroline=False)
fig.update_yaxes(gridcolor="#ecebe8", zeroline=False)

fig.update_layout(template="plotly_white",
    legend=dict(title=None, orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hoverlabel=dict(bgcolor="white"),
    height=550)

fig.show()


## Functions verifying an ML model
**4. (20 pts) Develop at least two functions that will verify fit or other attributes of analysis from a ML model. (You don't need to identify a specific model yet).  Include markup sections which explain why these functions will be useful in analyzing a ML model.**

In [131]:
# Function 1 evaluates the model accuracy
# MAE shows mean distance from each predicted value to the actual
# RMSE highlights outliers
# Rsquared analyzes fit

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluateModel(model, X_test, y_test):
  """ Given a model and the test data, evaluate the model's accuracy given 3 metrics. """
  y_pred = model.predict(X_test)

  print("MAE:", mean_absolute_error(y_test, y_pred))
  print("RMSE:", mean_squared_error(y_test, y_pred) ** 0.5)
  print("R²:", r2_score(y_test, y_pred))


Function 1 will evaluate the fit of a regression model for Question 2, where the model predicts the year an artwork was made. The mean absolute error (MAE) is the average distance between the actual and predicted values. It doesn't consider whether predictions are too high or too low, only how far off they are on average, and it is in the same units as the target (years), so an MAE of 150 means the model is off by about 150 years on average. The root mean squared error (RMSE) is also in years, but because it squares each error before averaging, it penalizes large errors much more heavily. Comparing the two is useful: if RMSE is much larger than MAE, the model is making a few very large mistakes (for example, dating an ancient object to the 1800s) rather than many small ones. R-squared shows goodness of fit by measuring the proportion of the variance in the target that the model explains, where 1 is a perfect fit and 0 is no better than always predicting the average year. Together, these metrics give a more well-rounded picture of the fit of the model.

In [132]:
# Function 2 evaluates a classification model which would help answer question 1
# classification_report shows precision, recall, and F1 for each class
# and confusion matrix shows which kinds of mistakes the model makes

from sklearn.metrics import classification_report, confusion_matrix

def evaluateClassifier(model, X_test, y_test):
  """ Given a classifier and the test data, evaluate how well it predicts whether a work is on display. """
  y_pred = model.predict(X_test)

  print(classification_report(y_test, y_pred, zero_division=0))
  print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))


Function 2 will evaluate the classification model for Question 1, which predicts whether or not a work is on display. Due to the way our data is spread where only 9% of the works are on display, regular measures of accuracy can be misleading. For example, a model that always predicts that a work is not on display will already be over 90% accurate. The first statement, classification report, shows precision of how many of the predicted works are actually on display, breaking it further down into recall and F1. The confusion matrix shows the number of right and wrong guesses, so we can see what kinds of mistakes the model makes.

## Ethical Implications the data and ML 
**5. (10 pts) Discuss potential ethical implications of the data and using ML to analyze it.**

Some ethical implications include sourcing issues where many of the artists are not alive to consent to data about their artwork being analyzed a machine learning model, or even if they are alright with their data being public at all. The actual use vs the understood use by either the museum or the artists of their art and/or data attributes may not align. Additionally, art contains many important contexts and motifs that a machine learning model may not be able to capture which could lead to misrepresentation or misinterpretation of art. Missing information especially from older art could lead to some bias in how the model is able to represent the sample, and historically underrepresented artists may continue to be underrepresented in the model's analysis.

## References

Bradley, K. (2015, January 23). Why museums hide masterpieces away. BBC Culture. [https://www.bbc.com/culture/article/20150123-7-masterpieces-you-cant-see](https://www.bbc.com/culture/article/20150123-7-masterpieces-you-cant-see)

Guerrilla Girls. (1989). Do women have to be naked to get into the Met. Museum? The Metropolitan Museum of Art, New York, NY. [https://www.metmuseum.org/art/collection/search/849438](https://www.metmuseum.org/art/collection/search/849438)

Solly, M. (2019, September 20). Study shows U.S. museums still lag when it comes to acquiring works by women artists. Smithsonian Magazine. [https://www.smithsonianmag.com/smart-news/survey-us-museums-reveals-few-gains-female-artists-past-decade-180973190/](https://www.smithsonianmag.com/smart-news/survey-us-museums-reveals-few-gains-female-artists-past-decade-180973190/)